# Model 3a: Pure SARIMA Model for Drug `R06`

## Hyperparameter Selection Methodology:
Seasonal order candidates $(p,d,q) 	imes (P,D,Q)_7$ evaluated on validation set.


In [1]:
# Dynamic Dependency Guard & Environment Initialization
import sys, subprocess, os

def install_and_import(pkg, module_name=None):
    if module_name is None:
        module_name = pkg
    try:
        __import__(module_name)
    except ImportError:
        print(f"Installing missing dependency: {pkg}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

install_and_import('numpy')
install_and_import('pandas')
install_and_import('matplotlib')
install_and_import('seaborn')
install_and_import('scikit-learn', 'sklearn')
install_and_import('statsmodels')
install_and_import('lightgbm')
install_and_import('xgboost')
install_and_import('shap')
install_and_import('prophet')
install_and_import('optuna')
install_and_import('torch')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style="whitegrid")
plt.rcParams.update({
    'font.sans-serif': 'Inter, Roboto, Arial, sans-serif',
    'axes.edgecolor': '#cccccc',
    'axes.linewidth': 1.0,
    'grid.color': '#eeeeee',
    'grid.linestyle': '--'
})

TARGET_DRUG = 'R06'
data_dir = r'c:\Users\ranje\sales forcasting\times_series\dataset'

train_df = pd.read_csv(os.path.join(data_dir, 'train_daily.csv'))
val_df   = pd.read_csv(os.path.join(data_dir, 'val_daily.csv'))
test_df  = pd.read_csv(os.path.join(data_dir, 'test_daily.csv'))

for df in [train_df, val_df, test_df]:
    df['date'] = pd.to_datetime(df['date'])

train_series = train_df.sort_values('date').set_index('date')[TARGET_DRUG].asfreq('D')
val_series   = val_df.sort_values('date').set_index('date')[TARGET_DRUG].asfreq('D')
test_series  = test_df.sort_values('date').set_index('date')[TARGET_DRUG].asfreq('D')

combined_series = pd.concat([train_series, val_series]).asfreq('D')
full_series     = pd.concat([combined_series, test_series]).asfreq('D')

def evaluate_metrics(y_true, y_pred):
    y_true = np.array(y_true, dtype=float)
    y_pred = np.clip(np.array(y_pred, dtype=float), 0, None)
    rmse  = np.sqrt(np.mean((y_true - y_pred)**2))
    mae   = np.mean(np.abs(y_true - y_pred))
    wape  = np.sum(np.abs(y_true - y_pred)) / np.sum(y_true) * 100
    rmsle = np.sqrt(np.mean((np.log1p(y_true) - np.log1p(y_pred))**2))
    return {'RMSLE': rmsle, 'RMSE': rmse, 'MAE': mae, 'WAPE (%)': wape}

print(f"Dataset for {TARGET_DRUG} loaded successfully!")
print(f"  * Train  : {train_series.index.min().strftime('%Y-%m-%d')} to {train_series.index.max().strftime('%Y-%m-%d')} ({len(train_series)} days)")
print(f"  * Val    : {val_series.index.min().strftime('%Y-%m-%d')} to {val_series.index.max().strftime('%Y-%m-%d')} ({len(val_series)} days)")
print(f"  * Test   : {test_series.index.min().strftime('%Y-%m-%d')} to {test_series.index.max().strftime('%Y-%m-%d')} ({len(test_series)} days)")


Dataset for R06 loaded successfully!
  * Train  : 2014-01-02 to 2017-12-31 (1460 days)
  * Val    : 2018-01-01 to 2018-12-31 (365 days)
  * Test   : 2019-01-01 to 2019-10-08 (281 days)


In [2]:
# Step 1: SARIMA Seasonal Order Search Code
from statsmodels.tsa.statespace.sarimax import SARIMAX

sarima_configs = [
    {'order': (1,1,1), 'seasonal_order': (1,1,1,7)},
    {'order': (1,1,1), 'seasonal_order': (0,1,1,7)},
    {'order': (2,1,1), 'seasonal_order': (1,1,1,7)}
]

best_sarima_cfg = None
best_val_rmsle = float('inf')

print("=== SARIMA Order Grid Search on 2018 Validation Set ===")
for cfg in sarima_configs:
    try:
        m = SARIMAX(np.log1p(train_series), **cfg, enforce_stationarity=False, enforce_invertibility=False).fit(disp=False)
        pred_log = m.forecast(steps=len(val_series))
        pred_val = np.clip(np.expm1(pred_log), 0, None)
        met = evaluate_metrics(val_series.values, pred_val)['RMSLE']
        print(f"  * Config {cfg} : Val RMSLE = {met:.6f}")
        if met < best_val_rmsle:
            best_val_rmsle = met
            best_sarima_cfg = cfg
    except Exception as e:
        continue

print(f"Selected Optimal SARIMA Configuration: {best_sarima_cfg}")


=== SARIMA Order Grid Search on 2018 Validation Set ===


  * Config {'order': (1, 1, 1), 'seasonal_order': (1, 1, 1, 7)} : Val RMSLE = 0.913132


  * Config {'order': (1, 1, 1), 'seasonal_order': (0, 1, 1, 7)} : Val RMSLE = 0.891021


  * Config {'order': (2, 1, 1), 'seasonal_order': (1, 1, 1, 7)} : Val RMSLE = 0.912191
Selected Optimal SARIMA Configuration: {'order': (1, 1, 1), 'seasonal_order': (0, 1, 1, 7)}


In [3]:
# Step 2: Fit Selected SARIMA Model & Forecast 2019 Test
m3a_model = SARIMAX(np.log1p(combined_series), **best_sarima_cfg, enforce_stationarity=False, enforce_invertibility=False)
m3a_fit = m3a_model.fit(disp=False)

pred_log = m3a_fit.forecast(steps=len(test_series))
m3a_test_pred = np.clip(np.expm1(pred_log), 0, None)
m3a_test_pred.index = test_series.index

test_metrics = evaluate_metrics(test_series, m3a_test_pred)
print(f"=== FINAL TEST HOLD-OUT METRICS (2019) — MODEL 3a: PURE SARIMA ===")
for k, v in test_metrics.items():
    print(f"  * {k:10s}: {v:.4f}")

pd.DataFrame({'date': test_series.index, 'pred_SARIMA': m3a_test_pred.values}).to_csv('m3_sarima_preds.csv', index=False)


=== FINAL TEST HOLD-OUT METRICS (2019) — MODEL 3a: PURE SARIMA ===
  * RMSLE     : 0.9321
  * RMSE      : 3.8610
  * MAE       : 2.9741
  * WAPE (%)  : 77.8449
